In [14]:
import tensorflow as tf

batch_size = 32
img_height = 224
img_width = 224

# Diviser les données en 80% train et 20% pour validation/test
train_val_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds) - val_size  # Reste pour test

val_ds = val_test_ds.take(val_size)  # Premier 50% pour validation
test_ds = val_test_ds.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds)}")
print(f"Nombre de batches dans val_ds: {len(val_ds)}")
print(f"Nombre de batches dans test_ds: {len(test_ds)}")

Found 11540 files belonging to 3 classes.
Using 9232 files for training.
Found 11540 files belonging to 3 classes.
Using 2308 files for validation.
Nombre de batches dans train_ds: 289
Nombre de batches dans val_ds: 36
Nombre de batches dans test_ds: 37


In [15]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_maiis = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset maiis',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size = int(0.5 * len(val_test_ds_maiis))  # 50% de val_test_ds pour validation
test_size = len(val_test_ds_maiis) - val_size  # Reste pour test

val_ds_maiis = val_test_ds_maiis.take(val_size)  # Premier 50% pour validation
test_ds_maiis = val_test_ds_maiis.skip(val_size)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_maiis)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_maiis)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_maiis)}")

Found 11480 files belonging to 3 classes.
Using 9184 files for training.
Found 11480 files belonging to 3 classes.
Using 2296 files for validation.
Nombre de batches dans train_ds: 287
Nombre de batches dans val_ds: 36
Nombre de batches dans test_ds: 36


In [16]:
## chargement dataset maiis
# Diviser les données en 80% train et 20% pour validation/test
train_val_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

val_test_ds_mixte = tf.keras.preprocessing.image_dataset_from_directory(
    'dataset mixte',
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size,
    label_mode='categorical')

# Diviser les 20% en 10% pour validation et 10% pour test
val_size_mixte = int(0.5 * len(val_test_ds_mixte))  # 50% de val_test_ds pour validation
test_size_mixte = len(val_test_ds_mixte) - val_size_mixte  # Reste pour test

val_ds_mixte = val_test_ds_maiis.take(val_size_mixte)  # Premier 50% pour validation
test_ds_mixte = val_test_ds_maiis.skip(val_size_mixte)  # Dernier 50% pour test

# Vérifier la répartition
print(f"Nombre de batches dans train_ds: {len(train_val_ds_mixte)}")
print(f"Nombre de batches dans val_ds: {len(val_ds_mixte)}")
print(f"Nombre de batches dans test_ds: {len(test_ds_mixte)}")

Found 11637 files belonging to 3 classes.
Using 9310 files for training.
Found 11637 files belonging to 3 classes.
Using 2327 files for validation.
Nombre de batches dans train_ds: 291
Nombre de batches dans val_ds: 36
Nombre de batches dans test_ds: 36


In [20]:
from tensorflow.keras.applications import MobileNetV3Large  # Utiliser MobileNetV3Large ou MobileNetV3Small
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping
from tensorflow.keras.models import load_model
import tensorflow as tf
from tensorflow.keras.optimizers import Adam

# Définir les dimensions d'image pour MobileNetV3
img_height, img_width = 224, 224

# Charger le modèle MobileNetV3 pré-entraîné
base_model = MobileNetV3Large(input_shape=(img_height, img_width, 3),
                              include_top=False,
                              weights='imagenet')
base_model.trainable = False  # Geler les poids du modèle pré-entraîné

# Débloquer les 50 dernières couches pour le fine-tuning
for layer in base_model.layers[-50:]:
    layer.trainable = True

# Ajouter une couche Global Average Pooling
global_average_layer = layers.GlobalAveragePooling2D()(base_model.output)

# Ajouter des couches fully connected supplémentaires avec Dropout
dense_1 = layers.Dense(1024, activation='relu')(global_average_layer)
dropout_1 = layers.Dropout(0.5)(dense_1)

dense_2 = layers.Dense(512, activation='relu')(dropout_1)
dropout_2 = layers.Dropout(0.5)(dense_2)

dense_3 = layers.Dense(256, activation='relu')(dropout_2)
dropout_3 = layers.Dropout(0.5)(dense_3)

dense_4 = layers.Dense(128, activation='relu')(dropout_3)
dropout_4 = layers.Dropout(0.5)(dense_4)

# Créer les sorties pour chaque nutriment (13 au total)
outputs = []
for nutrient in range(13):
    output = layers.Dense(3, activation='softmax', name=f'nutrient_{nutrient}')(dropout_4)
    outputs.append(output)

# Créer le modèle final avec MobileNetV3 en entrée et les 13 sorties en sortie
model = models.Model(inputs=base_model.input, outputs=outputs)

# Compilation du modèle avec seulement 'accuracy' comme métrique
#metrics = ['accuracy']
#metrics_list = [metrics] * 13  # Appliquer 'accuracy' à chaque nutriment

model.compile(optimizer=Adam(learning_rate=0.001),
              loss=['categorical_crossentropy'] * 13,
              metrics = ['accuracy'] * 13)


# Afficher un résumé du modèle pour vérifier les couches et les sorties
model.summary()

# Préparer les datasets (à adapter en fonction de vos données)
# train_dataset = ...  # Chargez vos données d'entraînement ici
# val_dataset = ...    # Chargez vos données de validation ici

# Ajout de callbacks pour la sauvegarde et le monitoring
# Entraîner le modèle



Model: "functional_6"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_6 (InputLayer)    │ (None, 224, 224, 3)       │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ rescaling_6 (Rescaling)       │ (None, 224, 224, 3)       │               0 │ input_layer_6[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv (Conv2D)                 │ (None, 112, 112, 16)      │             432 │ rescaling_6[0][0]          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ conv_bn (BatchNormalization)  │ (None, 112, 112, 16)      │              64 │ conv[0][0]                 │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ activation_120 (Activation)   │ (None, 112, 112, 16)      │               0 │ conv_bn[0][0]              │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise       │ (None, 112, 112, 16)      │             144 │ activation_120[0][0]       │
│ (DepthwiseConv2D)             │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_depthwise_bn    │ (None, 112, 112, 16)      │              64 │ expanded_conv_depthwise[0… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_114 (ReLU)              │ (None, 112, 112, 16)      │               0 │ expanded_conv_depthwise_b… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project         │ (None, 112, 112, 16)      │             256 │ re_lu_114[0][0]            │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_project_bn      │ (None, 112, 112, 16)      │              64 │ expanded_conv_project[0][… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_add (Add)       │ (None, 112, 112, 16)      │               0 │ activation_120[0][0],      │
│                               │                           │                 │ expanded_conv_project_bn[… │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_1_expand        │ (None, 112, 112, 64)      │           1,024 │ expanded_conv_add[0][0]    │
│ (Conv2D)                      │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ expanded_conv_1_expand_bn     │ (None, 112, 112, 64)      │             256 │ expanded_conv_1_expand[0]… │
│ (BatchNormalization)          │                           │                 │                            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ re_lu_115 (ReLU)              │ (None, 112, 112, 64)      │               

 Total params: 4,674,471 (17.83 MB)

 Trainable params: 3,857,807 (14.72 MB)

 Non-trainable params: 816,664 (3.12 MB)

In [5]:
from tensorflow.keras.callbacks import ModelCheckpoint, EarlyStopping

# Définir le checkpoint pour sauvegarder les meilleurs poids
checkpoint = ModelCheckpoint('MobileNet_V3_weights.keras',
                             monitor='val_accuracy',
                             verbose=1,
                             mode='max',
                             save_best_only=True)

# Early stopping pour arrêter l'entraînement si la validation stagne
early = EarlyStopping(monitor="val_loss",
                      mode="min",
                      restore_best_weights=True,
                      patience=5)

# Liste des callbacks
callbacks_list = [checkpoint, early]

In [21]:
import time
# Entraînement du modèle tout en mesurant le temps
start_time = time.time()

history = model.fit(
    train_val_ds,
    epochs=15,
    validation_data=val_ds
)

end_time = time.time()

# Afficher le temps d'entraînement
# = end_time - start_time
print(f"Temps d'apprentissage : {training_time} secondes")

# Calcul manuel du F1-score après l'entraînement
#precision = history.history['precision'][-1]
#recall = history.history['recall'][-1]
#f1 = 2 * (precision * recall) / (precision + recall + tf.keras.backend.epsilon()) 
#print(f'F1 Score: {f1:.4f}')

Epoch 1/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 445s 1s/step - loss: 0.8406 - nutrient_0_accuracy: 0.5679 - val_loss: 12.6267 - val_nutrient_0_accuracy: 0.6849
Epoch 2/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 407s 1s/step - loss: 0.4609 - nutrient_0_accuracy: 0.7612 - val_loss: 598.0336 - val_nutrient_0_accuracy: 0.4627
Epoch 3/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 406s 1s/step - loss: 0.4089 - nutrient_0_accuracy: 0.7954 - val_loss: 268.3606 - val_nutrient_0_accuracy: 0.4349
Epoch 4/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 409s 1s/step - loss: 0.4155 - nutrient_0_accuracy: 0.7857 - val_loss: 122.3097 - val_nutrient_0_accuracy: 0.5226
Epoch 5/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 406s 1s/step - loss: 0.3339 - nutrient_0_accuracy: 0.8294 - val_loss: 1.9454 - val_nutrient_0_accuracy: 0.8047
Epoch 6/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 405s 1s/step - loss: 0.3014 - nutrient_0_accuracy: 0.8528 - val_loss: 1.0104 - val_nutrient_0_accuracy: 0.7934
Epoch 7/15
289/289 ━━━━━━━━━━━━━━━━━━━━ 407s 1s/step - loss: 0.2869 - nutrient_0_accura

In [22]:
# Sauvegarder le modèle au format HDF5
model.save("model/MobileNetV3_4.h5")

In [29]:
# Sauvegarder le modèle au format .keras
model.save('model/MobileNetV3_4.keras')

In [30]:
import numpy as np
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image

def predict_nutrients_with_names_from_deployed_model(model_path, img_path, nutrient_names, class_labels, img_height=224, img_width=224):
    """
    Charger le modèle et prédire les classes pour chaque nutriment avec des noms de classes et de nutriments.

    Args:
        model_path (str): Chemin vers le fichier du modèle (.h5).
        img_path (str): Chemin vers l'image à prédire.
        nutrient_names (list): Liste des noms des nutriments (13 éléments).
        class_labels (list): Liste des noms des classes (3 éléments).
        img_height (int): Hauteur de l'image redimensionnée (par défaut 224).
        img_width (int): Largeur de l'image redimensionnée (par défaut 224).

    Returns:
        dict: Un dictionnaire contenant les prédictions pour chaque nutriment avec noms de classes.
    """
    # Charger le modèle déployé
    model = load_model(model_path)

    # Charger et prétraiter l'image
    img = image.load_img(img_path, target_size=(img_height, img_width))
    img_array = image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0)  # Ajouter une dimension batch
    img_array = img_array / 255.0  # Normaliser l'image entre 0 et 1

    # Obtenir les prédictions du modèle
    predictions = model.predict(img_array)

    # Décoder les prédictions pour chaque sortie
    nutrient_predictions = {}
    for i, pred in enumerate(predictions):
        predicted_index = np.argmax(pred)  # Obtenir l'indice de la classe avec la plus haute probabilité
        predicted_class = class_labels[predicted_index]  # Récupérer le nom de la classe
        nutrient_predictions[nutrient_names[i]] = predicted_class

    return nutrient_predictions

In [31]:
nutrient_names = [
    'Azote', 'Bore', 'Calcium', 'Chlore', 'Cuivre',
    'Fer', 'Magnésium', 'Manganèse', 'Molybdène',
    'Phosphore', 'Potassium', 'Souffre', 'Zinc'
]
class_labels = ['Légère', 'Saine', 'Sévère']

# Chemin vers une image à prédire
img_path = "IMG_1181.JPG"

# Chemins vers le modèle et l'image
model_path = 'model/MobileNetV3_4.keras'

# Prédictions avec noms des nutriments et des classes
predictions = predict_nutrients_with_names_from_deployed_model(model_path, img_path, nutrient_names, class_labels)

# Afficher les résultats
for nutrient, predicted_class in predictions.items():
    print(f"{nutrient}: {predicted_class}")


C:\Users\Bamba\AppData\Roaming\Python\Python312\site-packages\keras\src\saving\saving_lib.py:713: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 80 variables whereas the saved optimizer has 158 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


1/1 ━━━━━━━━━━━━━━━━━━━━ 3s 3s/step
Azote: Sévère
Bore: Saine
Calcium: Saine
Chlore: Saine
Cuivre: Légère
Fer: Sévère
Magnésium: Légère
Manganèse: Légère
Molybdène: Saine
Phosphore: Légère
Potassium: Sévère
Souffre: Sévère
Zinc: Saine


In [8]:
import tensorflow as tf
print(tf.__version__)

2.17.0


In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.metrics import Precision, Recall

# Recharger le modèle
model = load_model("model/saved_model", custom_objects={
    'Precision': Precision,
    'Recall': Recall
})

# Vérifier le résumé
model.summary()

In [7]:
model.save("model/MobileNetV3_04_11.h5")

In [8]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

37/37 ━━━━━━━━━━━━━━━━━━━━ 89s 2s/step - loss: 0.3828 - nutrient_0_accuracy: 0.8239 - nutrient_0_precision: 0.8239 - nutrient_0_recall: 0.8237
Nombre total de résultats: 4
Résultats de l'évaluation: [0.3959839940071106, 0.8183391094207764, 0.8181818127632141, 0.8174740672111511]
La structure des résultats est différente de celle attendue.


In [9]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_mixte)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

60/60 ━━━━━━━━━━━━━━━━━━━━ 144s 2s/step - loss: 4.0309 - nutrient_0_accuracy: 0.4268 - nutrient_0_precision: 0.4282 - nutrient_0_recall: 0.4160
Nombre total de résultats: 4
Résultats de l'évaluation: [3.818134069442749, 0.4173640310764313, 0.4173398017883301, 0.4053347408771515]
La structure des résultats est différente de celle attendue.


In [10]:
# Évaluation sur les données de validation
val_results = model.evaluate(test_ds_maiis)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

36/36 ━━━━━━━━━━━━━━━━━━━━ 73s 2s/step - loss: 3.8972 - nutrient_0_accuracy: 0.4055 - nutrient_0_precision: 0.4062 - nutrient_0_recall: 0.3960
Nombre total de résultats: 4
Résultats de l'évaluation: [3.8084464073181152, 0.41171327233314514, 0.41134113073349, 0.3994755148887634]
La structure des résultats est différente de celle attendue.


In [11]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

36/36 ━━━━━━━━━━━━━━━━━━━━ 70s 2s/step - loss: 0.4091 - nutrient_0_accuracy: 0.8312 - nutrient_0_precision: 0.8313 - nutrient_0_recall: 0.8312
Nombre total de résultats: 4
Résultats de l'évaluation: [0.38420361280441284, 0.8246527910232544, 0.8253692388534546, 0.8246527910232544]
La structure des résultats est différente de celle attendue.


In [12]:
# Évaluation sur les données de validation
val_results = model.evaluate(val_ds)

# Affichage de la longueur des résultats et des résultats eux-mêmes
print(f"Nombre total de résultats: {len(val_results)}")
print(f"Résultats de l'évaluation: {val_results}")

# Si vous avez plus ou moins de 13 nutriments, ajustez ici le nombre de nutriments
expected_nutrients = 13

# Vérifiez la structure exacte des résultats pour éviter l'IndexError
if len(val_results) >= expected_nutrients * 4 + 1:
    total_val_loss = val_results[0]
    nutrient_losses = val_results[1:expected_nutrients + 1]
    nutrient_accuracies = val_results[expected_nutrients + 1:expected_nutrients * 2 + 1]
    nutrient_precisions = val_results[expected_nutrients * 2 + 1:expected_nutrients * 3 + 1]
    nutrient_recalls = val_results[expected_nutrients * 3 + 1:expected_nutrients * 4 + 1]

    # Affichage des résultats globaux
    print(f"Total Validation Loss: {total_val_loss}")

    # Affichage des résultats pour chaque nutriment
    for i in range(expected_nutrients):
        print(f"Nutrient {i+1}:")
        print(f"  Validation Loss: {nutrient_losses[i]}")
        print(f"  Validation Accuracy: {nutrient_accuracies[i]}")
        print(f"  Precision: {nutrient_precisions[i]}")
        print(f"  Recall: {nutrient_recalls[i]}")
else:
    print("La structure des résultats est différente de celle attendue.")

36/36 ━━━━━━━━━━━━━━━━━━━━ 71s 2s/step - loss: 0.4173 - nutrient_0_accuracy: 0.8279 - nutrient_0_precision: 0.8286 - nutrient_0_recall: 0.8279
Nombre total de résultats: 4
Résultats de l'évaluation: [0.361061692237854, 0.8272569179534912, 0.8279756903648376, 0.8272569179534912]
La structure des résultats est différente de celle attendue.


In [13]:
import time
from tensorflow.keras import backend as K

# Fonction pour calculer la moyenne d'une métrique
def mean_metric(metric_values):
    return sum(metric_values) / len(metric_values)

# Nombre de nutriments
num_nutrients = 13

# Fonction pour calculer les moyennes pour chaque nutriment
def mean_nutrient_metric(metric_name, history):
    metrics = []
    for nutrient in range(num_nutrients):
        key = f'nutrient_{nutrient}_{metric_name}'
        if key in history:
            metrics.append(history[key])
    return [mean_metric(metric) for metric in zip(*metrics)]  # Moyenne sur les époques

# Calcul des moyennes pour l'ensemble des époques (entraînement)
mean_train_loss = mean_metric(history.history['loss'])
mean_train_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_train_precision = mean_nutrient_metric('precision', history.history)
mean_train_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (entraînement)
f1_train_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                   for p, r in zip(mean_train_precision, mean_train_recall)]
mean_f1_train = mean_metric(f1_train_scores)

# Calcul des moyennes pour l'ensemble des époques (validation)
mean_val_loss = mean_metric(history.history['val_loss'])
mean_val_accuracy = mean_nutrient_metric('accuracy', history.history)
mean_val_precision = mean_nutrient_metric('precision', history.history)
mean_val_recall = mean_nutrient_metric('recall', history.history)

# Calcul du F1-score moyen pour l'ensemble des époques (validation)
f1_val_scores = [2 * (p * r) / (p + r + K.epsilon()) 
                 for p, r in zip(mean_val_precision, mean_val_recall)]
mean_f1_val = mean_metric(f1_val_scores)

# Affichage des résultats
print(f"Moyenne de la perte sur l'ensemble d'entraînement : {mean_train_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble d'entraînement : {mean_metric(mean_train_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble d'entraînement : {mean_metric(mean_train_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble d'entraînement : {mean_metric(mean_train_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble d'entraînement : {mean_f1_train:.4f}")

print(f"Moyenne de la perte sur l'ensemble de validation : {mean_val_loss:.4f}")
print(f"Moyenne de l'accuracy sur l'ensemble de validation : {mean_metric(mean_val_accuracy):.4f}")
print(f"Moyenne de la précision sur l'ensemble de validation : {mean_metric(mean_val_precision):.4f}")
print(f"Moyenne du rappel sur l'ensemble de validation : {mean_metric(mean_val_recall):.4f}")
print(f"Moyenne du F1-score sur l'ensemble de validation : {mean_f1_val:.4f}")

Moyenne de la perte sur l'ensemble d'entraînement : 0.3479
Moyenne de l'accuracy sur l'ensemble d'entraînement : 0.8267
Moyenne de la précision sur l'ensemble d'entraînement : 0.8344
Moyenne du rappel sur l'ensemble d'entraînement : 0.8132
Moyenne du F1-score sur l'ensemble d'entraînement : 0.8232
Moyenne de la perte sur l'ensemble de validation : 40.0045
Moyenne de l'accuracy sur l'ensemble de validation : 0.8267
Moyenne de la précision sur l'ensemble de validation : 0.8344
Moyenne du rappel sur l'ensemble de validation : 0.8132
Moyenne du F1-score sur l'ensemble de validation : 0.8232
